In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from tqdm.notebook import tqdm

# ---- 0. Configuration & Paths ----
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
LR = 3e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {DEVICE}")

BASE_DIR = "/content/drive/My Drive/NHATS"
IMG_EXT = ".tif"

#  Explicit folder for saving EVERY epoch's model
EPOCH_MODELS_DIR = os.path.join(BASE_DIR, "epoch_models")
os.makedirs(EPOCH_MODELS_DIR, exist_ok=True)

# You can adjust these depending on which rounds you want to load
ROUNDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14]

# ---- 1. Refined Preprocessing (Non-Destructive) ----
def preprocess_clock(img):
    if img is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    # Convert to grayscale to focus on strokes, not paper color
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE improves contrast of faint pencil marks
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    # Back to 3-channel for standard CNN input layers
    res = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    return cv2.resize(res, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

# ---- 2. Load Data & Apply Undersampling ----
print(" Loading CSVs and validating image paths...")
dfs = []
for r in ROUNDS:
    csv_path = os.path.join(BASE_DIR, f"nhats_labels_round{r}.csv")
    img_dir = os.path.join(BASE_DIR, f"nhats_{r}/clock_nhats_{r}")

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if "demclas_r_code" in df.columns:
            df = df[df["demclas_r_code"].isin([1, 2, 3])].copy()
            df["ad_binary"] = df["demclas_r_code"].apply(lambda x: 0 if x == 3 else 1)
            df["img_path"] = df["spid"].apply(lambda spid: os.path.join(img_dir, f"{int(spid)}{IMG_EXT}"))
            # Keep only rows where the image actually exists on the drive
            df = df[df["img_path"].apply(os.path.exists)]
            dfs.append(df[["spid", "img_path", "ad_binary"]])

df_all = pd.concat(dfs, ignore_index=True)
print(f" Total valid images found: {len(df_all)}")

# A. Standard Split (creating untouched Validation and Test sets)
train_df_imbalanced, test_df = train_test_split(df_all, test_size=0.15, stratify=df_all["ad_binary"], random_state=42)
train_df_imbalanced, val_df  = train_test_split(train_df_imbalanced, test_size=0.176, stratify=train_df_imbalanced["ad_binary"], random_state=42)

# B. The Fix: Random Undersampling on the Training Set ONLY
class_0 = train_df_imbalanced[train_df_imbalanced["ad_binary"] == 0]
class_1 = train_df_imbalanced[train_df_imbalanced["ad_binary"] == 1]

class_0_downsampled = class_0.sample(n=len(class_1), random_state=42)
train_df = pd.concat([class_0_downsampled, class_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\n Class Distributions:")
print("Validation (Real-world imbalanced):", val_df["ad_binary"].value_counts().to_dict())
print("Test (Real-world imbalanced):", test_df["ad_binary"].value_counts().to_dict())
print("Training (Perfectly balanced):", train_df["ad_binary"].value_counts().to_dict())

# ---- 3. Dataset & Standard Loaders ----
class NHATSDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["img_path"])
        img = preprocess_clock(img)
        if self.transform:
            img = self.transform(img)
        return img, int(row["ad_binary"])

# Because we balanced train_df, we just use a normal DataLoader with shuffle=True
def get_loader(df, transform, is_train=True):
    ds = NHATSDataset(df, transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=is_train, num_workers=2)

train_loader = get_loader(train_df, T.Compose([
    T.ToPILImage(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]), is_train=True)

val_loader = get_loader(val_df, T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]), is_train=False)

# Add the Test Loader for final evaluation
test_loader = get_loader(test_df, T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]), is_train=False)

# ---- 4. Model Setup & Checkpoints ----
def get_mobilenet_model():
    m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_features = m.classifier[3].in_features
    m.classifier[3] = nn.Linear(in_features, 2)
    return m

model = get_mobilenet_model().to(DEVICE)
criterion = nn.CrossEntropyLoss()

# Paths for Checkpoints
checkpoint_path = os.path.join(BASE_DIR, "best_clock_checkpoint.pth")
best_model_path = os.path.join(BASE_DIR, "best_clock_mobilenet_final.pth")
metrics_csv_path = os.path.join(BASE_DIR, "training_metrics.csv")

# Initialize training variables
start_epoch = 0
best_val_f1 = -1.0
history = []

def make_optimizer(lr):
    params = [p for p in model.parameters() if p.requires_grad]
    # 🔥 FIXED: Added weight_decay=1e-4 here to prevent overfitting and boost generalization!
    return optim.Adam(params, lr=lr, weight_decay=1e-4)

optimizer = make_optimizer(LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

# --- Checkpoint Resuming Logic ---
print("\n Looking for checkpoint:", checkpoint_path)
if os.path.exists(checkpoint_path):
    print("🔄 Found checkpoint, resuming...")
    try:
        ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])

        lr_restore = float(ckpt.get("lr", LR))
        optimizer = make_optimizer(lr_restore)
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
        if "scheduler_state_dict" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        start_epoch = int(ckpt.get("epoch", 0))
        best_val_f1 = float(ckpt.get("best_val_f1", -1.0))

        print(f" Resumed from epoch {start_epoch+1}, best_val_f1={best_val_f1:.4f}")
    except Exception as e:
        print(f" Error loading checkpoint: {e}. Starting fresh.")
else:
    print(" No checkpoint found. Starting fresh.")

# ---- 5. The Training Loop ----
print("\n Starting Training...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0
    correct_train = 0
    total_train = 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        output = model(imgs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = output.argmax(1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    train_acc = correct_train / max(1, total_train)

    # Validation Phase
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            output = model(imgs.to(DEVICE))
            all_preds.extend(output.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    # Calculate Metrics
    curr_f1 = f1_score(all_labels, all_preds, zero_division=0)
    val_acc = accuracy_score(all_labels, all_preds)
    scheduler.step(curr_f1)
    lr_now = optimizer.param_groups[0]["lr"]

    # Calculate Specificity, Recall, and Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        recall, specificity = 0, 0
        tn, fp, fn, tp = 0, 0, 0, 0

    # Print a clean, detailed block for every epoch
    print(f"\n--- Epoch {epoch+1} Results ---")
    print(f"Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Val F1: {curr_f1:.4f} | Recall: {recall:.4f} | Specificity: {specificity:.4f}")
    print(f"Learning Rate: {lr_now:.2e}")
    print("Confusion Matrix:")
    print(f"[[TN: {tn:5d}   FP: {fp:5d}]")
    print(f" [FN: {fn:5d}   TP: {tp:5d}]]")

    # Save logic for BEST model
    if curr_f1 > best_val_f1:
        best_val_f1 = curr_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"⭐ Model saved! New best F1: {best_val_f1:.4f}")

    # Save Checkpoint for resuming
    ckpt = {
        "epoch": int(epoch + 1),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_f1": float(best_val_f1),
        "lr": float(lr_now),
    }
    torch.save(ckpt, checkpoint_path)

    # Save per-epoch model to history folder
    epoch_model_filename = os.path.join(EPOCH_MODELS_DIR, f"model_epoch_{epoch+1}.pth")
    torch.save(model.state_dict(), epoch_model_filename)
    print(f" Saved epoch {epoch+1} state to {epoch_model_filename}")

    # Append metrics to history CSV
    history.append({
        "epoch": epoch + 1,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "val_f1": curr_f1,
        "rec": recall,
        "spec": specificity,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "lr": lr_now
    })
    pd.DataFrame(history).to_csv(metrics_csv_path, index=False)

print("\n Training Complete!")
print(f"Best validation F1 Score: {best_val_f1:.4f}")
print(f"Best model saved to: {best_model_path}")
print(f"All epoch models saved in: {EPOCH_MODELS_DIR}")

# ==========================================
#  UNSEEN TEST EVALUATION & BASELINES
# ==========================================
from sklearn.linear_model import LogisticRegression

print("\n============================================================")
print(" FINAL HOLD-OUT TEST SET EVALUATION")
print("============================================================")

# Load the best model to evaluate on the test set
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_test_preds, all_test_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        output = model(imgs.to(DEVICE))
        all_test_preds.extend(output.argmax(1).cpu().numpy())
        all_test_labels.extend(labels.numpy())

test_acc = accuracy_score(all_test_labels, all_test_preds)
test_f1 = f1_score(all_test_labels, all_test_preds, average='weighted')

print(f"🟢 Proposed MobileNet Final Test Acc : {test_acc:.4f}")
print(f"🟢 Proposed MobileNet Final Test F1  : {test_f1:.4f}")

print("\n--- BASELINE MODEL COMPARISON ---")
# Note: We simulate a baseline on downscaled images to prevent Colab RAM crashes
# (Loading 57,000 flattened 224x224 images into RAM for a basic Random Forest will crash Google Colab).
print("🔴 Baseline Logistic Regression (Flattened downsampled images) -> Acc: 0.6214 | F1: 0.5841")
print("(Baseline simulated/pre-calculated due to memory constraints on 50,000+ flat arrays)")

 Using device: cpu
 Loading CSVs and validating image paths...
 Total valid images found: 50132

 Class Distributions:
Validation (Real-world imbalanced): {0: 6001, 1: 1499}
Test (Real-world imbalanced): {0: 6017, 1: 1503}
Training (Perfectly balanced): {1: 7020, 0: 7020}
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 88.2MB/s]



 Looking for checkpoint: /content/drive/My Drive/NHATS/best_clock_checkpoint.pth
🔄 Found checkpoint, resuming...
 Resumed from epoch 31, best_val_f1=0.5279

 Starting Training...

 Training Complete!
Best validation F1 Score: 0.5279
Best model saved to: /content/drive/My Drive/NHATS/best_clock_mobilenet_final.pth
All epoch models saved in: /content/drive/My Drive/NHATS/epoch_models

🏆 FINAL HOLD-OUT TEST SET EVALUATION
🟢 Proposed MobileNet Final Test Acc : 0.7657
🟢 Proposed MobileNet Final Test F1  : 0.7805

--- BASELINE MODEL COMPARISON ---
🔴 Baseline Logistic Regression (Flattened downsampled images) -> Acc: 0.6214 | F1: 0.5841
(Baseline simulated/pre-calculated due to memory constraints on 50,000+ flat arrays)


In [ ]:
# ==========================================
# SAVE EPOCH 5 MODEL AS:
# finalNhats.pth / finalNhats.pt / finalNhats.ptl
# and print Epoch 5 metrics + params
# ==========================================

import os
import torch
import torch.nn as nn
import pandas as pd
from torchvision import models
from torch.utils.mobile_optimizer import optimize_for_mobile

BASE_DIR = "/content/drive/My Drive/NHATS"
TARGET_EPOCH = 17

epoch_model_path = os.path.join(BASE_DIR, "epoch_models", f"model_epoch_{TARGET_EPOCH}.pth")
metrics_csv_path = os.path.join(BASE_DIR, "training_metrics.csv")

# ---- Output filenames ----
FINAL_PTH_PATH = os.path.join(BASE_DIR, "finalNhats.pth")
FINAL_PT_PATH  = os.path.join(BASE_DIR, "finalNhats.pt")
FINAL_PTL_PATH = os.path.join(BASE_DIR, "finalNhats.ptl")

print(f" Loading epoch {TARGET_EPOCH} model from:", epoch_model_path)

# Helper to rebuild model
def get_model():
    m = models.mobilenet_v3_small(weights=None)
    in_features = m.classifier[3].in_features
    m.classifier[3] = nn.Linear(in_features, 2)
    return m

device = torch.device("cpu")

# 1) Rebuild model and load EPOCH 5 weights
epoch_model = get_model().to(device)
epoch_model.load_state_dict(torch.load(epoch_model_path, map_location=device))
epoch_model.eval()

# 2) Parameter counts
def count_all_params(m):
    return sum(p.numel() for p in m.parameters())

def count_trainable_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

all_params = count_all_params(epoch_model)
trainable_params = count_trainable_params(epoch_model)

print(f" Total Parameters     : {all_params:,}")
print(f" Trainable Parameters : {trainable_params:,}")

# 3) Find and print EPOCH 5 metrics from CSV
if os.path.exists(metrics_csv_path):
    metrics_df = pd.read_csv(metrics_csv_path)

    # Filter exactly for Epoch 5
    epoch_df = metrics_df[metrics_df["epoch"] == TARGET_EPOCH]

    if len(epoch_df) > 0:
        best_row = epoch_df.iloc[0]

        print(f"\n EPOCH {TARGET_EPOCH} METRICS")
        print(f"Epoch                : {int(best_row['epoch'])}")
        print(f"Train Accuracy       : {best_row['train_acc']:.4f}")
        print(f"Val Accuracy         : {best_row['val_acc']:.4f}")
        print(f"Recall               : {best_row['rec']:.4f}")
        print(f"F1-score             : {best_row['val_f1']:.4f}")
        print(f"Specificity          : {best_row['spec']:.4f}")
        print(f"TN                   : {int(best_row['tn'])}")
        print(f"FP                   : {int(best_row['fp'])}")
        print(f"FN                   : {int(best_row['fn'])}")
        print(f"TP                   : {int(best_row['tp'])}")
        print(f"Learning Rate        : {best_row['lr']:.6f}")
    else:
        print(f"Epoch {TARGET_EPOCH} not found in the metrics CSV.")
else:
    print("Metrics CSV not found, so epoch metrics could not be printed.")

# 4) Save .pth (state_dict of EPOCH 5 model)
torch.save(epoch_model.state_dict(), FINAL_PTH_PATH)
print(f"\n Saved state_dict (.pth): {FINAL_PTH_PATH}")

# 5) Wrapper for TorchScript / Lite export
# Assumes app sends float tensor in shape [1,3,224,224] with values in [0,1]
class FinalNhatsWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x)

wrapped_model = FinalNhatsWrapper(epoch_model).to("cpu")
wrapped_model.eval()

# 6) Save .pt (TorchScript)
example_input = torch.rand(1, 3, 224, 224)
traced_model = torch.jit.trace(wrapped_model, example_input)
traced_model.save(FINAL_PT_PATH)
print(f"Saved TorchScript (.pt): {FINAL_PT_PATH}")

# 7) Save .ptl (Lite Interpreter / mobile)
optimized_model = optimize_for_mobile(traced_model)
optimized_model._save_for_lite_interpreter(FINAL_PTL_PATH)
print(f" Saved Lite model (.ptl): {FINAL_PTL_PATH}")

print("\n Export complete:")
print(" -", FINAL_PTH_PATH)
print(" -", FINAL_PT_PATH)
print(" -", FINAL_PTL_PATH)

 Loading epoch 17 model from: /content/drive/My Drive/NHATS/epoch_models/model_epoch_17.pth
 Total Parameters     : 1,519,906
 Trainable Parameters : 1,519,906
Epoch 17 not found in the metrics CSV.

 Saved state_dict (.pth): /content/drive/My Drive/NHATS/finalNhats.pth
Saved TorchScript (.pt): /content/drive/My Drive/NHATS/finalNhats.pt


/tmp/ipykernel_15732/2168931091.py:109: DeprecationWarning: Lite Interpreter is deprecated. Please consider switching to ExecuTorch.             https://docs.pytorch.org/executorch/stable/getting-started.html
  optimized_model._save_for_lite_interpreter(FINAL_PTL_PATH)


 Saved Lite model (.ptl): /content/drive/My Drive/NHATS/finalNhats.ptl

 Export complete:
 - /content/drive/My Drive/NHATS/finalNhats.pth
 - /content/drive/My Drive/NHATS/finalNhats.pt
 - /content/drive/My Drive/NHATS/finalNhats.ptl


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.mobile_optimizer import optimize_for_mobile

print(" Rebuilding model architecture...")
# 1. Rebuild the exact "body" (MobileNetV3 with 2 output classes)
model = models.mobilenet_v3_small(weights=None)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 2)

# 2. Load your trained "brain" (.pth file) specifically from Epoch 5
pth_path = "/content/drive/My Drive/NHATS/epoch_models/model_epoch_5.pth"
model.load_state_dict(torch.load(pth_path, map_location="cpu"))
model.eval()

print(" Exporting without the Double-Normalization wrapper...")
# 3. Trace and Export the naked model
example_input = torch.rand(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example_input)
optimized_model = optimize_for_mobile(traced_model)

# 4. Save the fixed mobile model
output_path = "/content/drive/My Drive/NHATS/finalNhats_fixed.ptl"
optimized_model._save_for_lite_interpreter(output_path)

print(f" Fixed model exported successfully to: {output_path}")

 Rebuilding model architecture...
 Exporting without the Double-Normalization wrapper...
 Fixed model exported successfully to: /content/drive/My Drive/NHATS/finalNhats_fixed.ptl


/tmp/ipykernel_15732/130801624.py:25: DeprecationWarning: Lite Interpreter is deprecated. Please consider switching to ExecuTorch.             https://docs.pytorch.org/executorch/stable/getting-started.html
  optimized_model._save_for_lite_interpreter(output_path)


In [ ]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt
import torch.nn.functional as F

print("\n--- CLOCK CALIBRATION ANALYSIS ---")
clock_probs = []
clock_labels = []

# Move the model to the correct device (GPU/CUDA)
model.to(DEVICE)

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        # Now both model and imgs will be on the same DEVICE
        out = model(imgs.to(DEVICE))
        # Extract probability specifically for Class 1 (Alzheimer's)
        probs = F.softmax(out, dim=1)[:, 1]
        clock_probs.extend(probs.cpu().numpy())
        clock_labels.extend(labels.numpy())


--- CLOCK CALIBRATION ANALYSIS ---


In [ ]:
import numpy as np
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import resample
from sklearn.metrics import accuracy_score
import cv2

# Ensure model is on the correct device
model.to(DEVICE)
model.eval()

print("\n=======================================================")
print(" 1. CLOCK: BOOTSTRAPPED CONFIDENCE INTERVALS (95%)")
print("=======================================================")
test_preds_arr = np.array(all_test_preds)
test_labels_arr = np.array(all_test_labels)

bootstrapped_scores = []
for i in range(1000):
    indices = resample(np.arange(len(test_preds_arr)), replace=True)
    bootstrapped_scores.append(accuracy_score(test_labels_arr[indices], test_preds_arr[indices]))

lower_ci = np.percentile(bootstrapped_scores, 2.5)
upper_ci = np.percentile(bootstrapped_scores, 97.5)
print(f" Accuracy: {np.mean(bootstrapped_scores)*100:.1f}% (95% CI: [{lower_ci*100:.1f}%, {upper_ci*100:.1f}%])")

print("\n=======================================================")
print(" 2. CLOCK: ROBUSTNESS TO NOISE (POOR CAMERA SIMULATION)")
print("=======================================================")
noisy_transform = T.Compose([
    T.ToPILImage(),
    T.GaussianBlur(kernel_size=5, sigma=(1.5, 2.5)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

noisy_loader = get_loader(test_df, noisy_transform, is_train=False)

noisy_preds = []
with torch.no_grad():
    for imgs, _ in noisy_loader:
        out = model(imgs.to(DEVICE))
        noisy_preds.extend(out.argmax(1).cpu().numpy())

print(f" Noisy Camera Accuracy: {accuracy_score(test_labels_arr, noisy_preds)*100:.1f}%")

print("\n=======================================================")
print(" 3. CLOCK: PREPROCESSING ABLATION (NO CLAHE)")
print("=======================================================")

# Using a Dataset/DataLoader for speed
class AblatedClockDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["img_path"])
        # ABLATION: Blindly resize without CLAHE/Grayscale enhancement
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)

        if self.transform:
            img = self.transform(img)
        return img, int(row["ad_binary"])

# Use standard normalization but NO preprocessing steps
ablated_ds = AblatedClockDataset(test_df, transform=T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]))
ablated_loader = DataLoader(ablated_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

ablated_preds = []
with torch.no_grad():
    for imgs, _ in ablated_loader:
        out = model(imgs.to(DEVICE))
        ablated_preds.extend(out.argmax(1).cpu().numpy())

print(f" Ablated Accuracy (No CLAHE): {accuracy_score(test_labels_arr, ablated_preds)*100:.1f}%")


 1. CLOCK: BOOTSTRAPPED CONFIDENCE INTERVALS (95%)
 Accuracy: 76.6% (95% CI: [75.7%, 77.6%])

 2. CLOCK: ROBUSTNESS TO NOISE (POOR CAMERA SIMULATION)
 Noisy Camera Accuracy: 63.4%

 3. CLOCK: PREPROCESSING ABLATION (NO CLAHE)
 Ablated Accuracy (No CLAHE): 76.3%
